# 02 — 数据预处理

本 Notebook 用于数据清洗和特征工程。

## 目标
- 缺失值处理
- 异常值检测与处理
- 类别特征编码
- 特征缩放（标准化/归一化）
- 训练集 / 测试集划分
- 保存处理后的数据到 `data/processed/`

In [1]:
# %% [markdown]
# # 02 — 数据预处理
# 本 Notebook 用于数据清洗和特征工程。
# 
# ## 目标
# 1. 缺失值处理
# 2. 异常值检测与处理
# 3. 类别特征编码
# 4. 特征缩放（标准化/归一化）
# 5. 训练集 / 测试集划分
# 6. 保存处理后的数据到 data/processed/

# %% [markdown]
# ## 1. 导入库
# %%
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# 自动创建输出目录
import os
os.makedirs("../data/processed", exist_ok=True)

# %% [markdown]
# ## 2. 加载原始数据
# %%
df = pd.read_csv("../data/raw/student-mat.csv", sep=";")
print(f"✅ 数据加载成功，共 {df.shape[0]} 行，{df.shape[1]} 列")

# %% [markdown]
# ## 3. 缺失值处理
# %%
missing_count = df.isnull().sum().sum()
if missing_count == 0:
    print("✅ 数据集中无缺失值，无需处理")
else:
    numeric_cols = df.select_dtypes(include=np.number).columns
    categorical_cols = df.select_dtypes(include='object').columns
    
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    df[categorical_cols] = df[categorical_cols].fillna(df[categorical_cols].mode().iloc[0])
    print(f"✅ 已处理 {missing_count} 个缺失值")

# %% [markdown]
# ## 4. 异常值检测与处理
# %%
absences_95 = df["absences"].quantile(0.95)
df["absences"] = df["absences"].clip(upper=absences_95)
print(f"✅ 已对absences特征进行截断处理，上限设为 {absences_95}")

print("ℹ️ 其他特征的异常值已保留，因为它们包含重要的预测信息")

# %% [markdown]
# ## 5. 类别特征编码
# %%
# 分离特征和目标变量
X = df.drop("G3", axis=1)
y = df["G3"]

# 区分数值特征和类别特征
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

print(f"📋 数值特征数量: {len(numeric_features)}")
print(f"📋 类别特征数量: {len(categorical_features)}")

# 创建预处理管道
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse=False), categorical_features)
    ])

# %% [markdown]
# ## 6. 训练集 / 测试集划分
# %%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"✅ 训练集大小: {X_train.shape[0]} 行")
print(f"✅ 测试集大小: {X_test.shape[0]} 行")

# %% [markdown]
# ## 7. 特征缩放与编码
# %%
# 拟合训练集并转换
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 获取特征名称
feature_names = (
    numeric_features + 
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
)

print(f"✅ 预处理完成，特征数量从 {X.shape[1]} 变为 {len(feature_names)}")

# %% [markdown]
# ## 8. 保存处理后的数据
# %%
# 转换为DataFrame保存
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)
y_train_df = pd.DataFrame(y_train, columns=["G3"], index=y_train.index)
y_test_df = pd.DataFrame(y_test, columns=["G3"], index=y_test.index)

# 保存到文件
X_train_df.to_csv("../data/processed/X_train.csv")
X_test_df.to_csv("../data/processed/X_test.csv")
y_train_df.to_csv("../data/processed/y_train.csv")
y_test_df.to_csv("../data/processed/y_test.csv")

# 保存预处理管道
import joblib
joblib.dump(preprocessor, "../data/processed/preprocessor.pkl")

print("✅ 所有处理后的数据已保存到 ../data/processed/ 目录")
print("✅ 预处理管道已保存为 preprocessor.pkl")

✅ 数据加载成功，共 395 行，33 列
✅ 数据集中无缺失值，无需处理
✅ 已对absences特征进行截断处理，上限设为 18.299999999999955
ℹ️ 其他特征的异常值已保留，因为它们包含重要的预测信息
📋 数值特征数量: 15
📋 类别特征数量: 17
✅ 训练集大小: 316 行
✅ 测试集大小: 79 行
✅ 预处理完成，特征数量从 32 变为 41
✅ 所有处理后的数据已保存到 ../data/processed/ 目录
✅ 预处理管道已保存为 preprocessor.pkl
